# 05.P · Mini-project: a tuned, explained churn model

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/05-powerful-models/project/project.ipynb)

Read the [brief and rubric](README.md) first. Fill in every `# TODO` cell. The setup cell is done for you. Check [solution.ipynb](solution.ipynb) only after you have a complete answer of your own.

## 0. Setup: the churn table from Module 03

In [ ]:
import os, logging
os.environ["PYTHONWARNINGS"] = "ignore"
import optuna; optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail
from sklearn.model_selection import train_test_split

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

cutoff = pd.Timestamp("2011-09-01")                       # features before, target after
past = sales[sales["InvoiceDate"] < cutoff]
future = sales[(sales["InvoiceDate"] >= cutoff) & (sales["InvoiceDate"] < "2011-12-01")]

cust = past.groupby("Customer ID").agg(orders=("Invoice", "nunique"), spend=("Revenue", "sum"),
                                       products=("StockCode", "nunique"),
                                       first_order=("InvoiceDate", "min"), last_order=("InvoiceDate", "max"))
cust["tenure_days"] = (cutoff - cust["first_order"]).dt.days
cust["days_since_last"] = (cutoff - cust["last_order"]).dt.days
future_spend = future.groupby("Customer ID")["Revenue"].sum().reindex(cust.index).fillna(0)
cust["churned"] = (future_spend == 0).astype(int)        # 1 = did not come back
cust = cust.drop(columns=["first_order", "last_order"])

features = ["orders", "spend", "products", "tenure_days", "days_since_last"]
X_train, X_test, y_train, y_test = train_test_split(
    cust[features], cust["churned"], test_size=0.25, random_state=42, stratify=cust["churned"])
print(f"{len(cust):,} customers, churn rate {cust['churned'].mean():.1%}, "
      f"train {len(X_train):,} / test {len(X_test):,}")

def expected_cost(y, p, t, c_voucher=5, c_lost=60):
    """Voucher for everyone with p >= t; a missed churner costs c_lost."""
    y, flag = np.asarray(y), np.asarray(p) >= t
    return c_voucher * flag.sum() + c_lost * ((y == 1) & ~flag).sum()

from pathlib import Path
def models_dir():
    """<repo>/models when run inside the course repo, ./models otherwise (e.g. Colab)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "data" / "download.py").exists():
            d = p / "models"; break
    else:
        d = Path("models")
    d.mkdir(exist_ok=True)
    return d

import lightgbm as lgb, shap, joblib, datetime, sklearn
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_val_predict, RandomizedSearchCV
from sklearn.metrics import roc_auc_score
from scipy.stats import loguniform, randint
cv = StratifiedKFold(5, shuffle=True, random_state=42)

## 1. Baseline

Fit the Module 03 baseline: `StandardScaler` + `LogisticRegression` in a pipeline. Report its 5-fold CV AUC on the training set (keep the per-fold scores) and its test AUC.

In [ ]:
# TODO

## 2. Tune a boosted model

Tune `lgb.LGBMClassifier(n_estimators=200, verbose=-1, random_state=42, n_jobs=1)` with `RandomizedSearchCV` (30 iterations, the same `cv`, `scoring="roc_auc"`) over `learning_rate` (log-uniform 0.005–0.3), `num_leaves` (2–63), `min_child_samples` (5–99) and `subsample`/`colsample_bytree` (0.5–1.0, set `subsample_freq=1`). Report the best settings and CV AUC. Do not touch the test set.

In [ ]:
# TODO

## 3. Is the difference real?

Compute the tuned model's per-fold CV AUC on the **same folds** as the baseline. Show the five fold-by-fold differences, their mean and std. Is the gain larger than the noise? Then open the test set once for both models.

In [ ]:
# TODO

## 4. Threshold by voucher cost

For both models, choose the cheapest threshold (£5 voucher, £60 lost customer) using cross-validated **training** predictions over thresholds 0.05–0.95. Then report the test cost of each model at its threshold, plus 'voucher to everyone' and 'voucher to nobody'.

In [ ]:
# TODO

## 5. Explain the model globally

Use `shap.TreeExplainer` on the tuned LightGBM. Draw the beeswarm for the test set and print mean |SHAP| per feature. Write two sentences on what drives churn risk.

In [ ]:
# TODO

## 6. Explain one customer

Pick a test customer whose LightGBM probability is close to the 75th percentile (a 'worried but not lost' customer). Draw their waterfall and print a three-line plain-English explanation: average risk, their risk, top three reasons.

In [ ]:
# TODO

## 7. Choose, save, document

Apply the one-standard-error rule (05.10) to the two models' CV scores and pick the final model. Save a bundle (pipeline, features, threshold, metrics, date, versions) to `models/` with a versioned name, reload it, and score one new customer. Print a model card.

In [ ]:
# TODO

### Model card / CEO summary

Write a one-page summary in plain English for the CEO: what it does, how good it is, which model and why, what drives churn, what you recommend (with £ figures), limits.

*Your summary here.*